# Phase 11: MLOps and experiment tracking

Run an engine-grouped hyperparameter search, then log its configuration, cross-validation score, trial table, and fitted estimator to a local MLflow tracking store. MLflow's UI can be opened with `mlflow ui --backend-store-uri ./mlruns`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.features.feature_engineering import build_feature_matrix
from src.features.target_engineering import prepare_fd_dataset
from src.models.hyperparameter_optimization import optimize_hyperparameters
from src.tracking.experiment_tracking import log_experiment
from src.utils.config import load_config

config = load_config()
project_root = Path('..').resolve()
tracking_config = config['tracking']
raw_dir = project_root / 'data/raw/CMAPSSData'
fd_data = prepare_fd_dataset('FD001', raw_dir=raw_dir)
feature_matrix = build_feature_matrix(fd_data['train'], target_column='rul_uncapped', windows=(3, 5, 10))

optimization = optimize_hyperparameters(
    feature_matrix,
    model_name='random_forest',
    target_column='rul_uncapped',
    group_column=config['ml']['group_column'],
    metric='rmse',
    n_splits=3,
    n_trials=30,
    random_state=config['project']['random_seed'],
)
report_dir = project_root / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
trials_path = report_dir / 'phase11_optimization_trials.csv'
optimization.trials.to_csv(trials_path, index=False)

run_id = log_experiment(
    experiment_name=tracking_config['experiment_name'],
    run_name='FD001-random-forest-tuning',
    params={
        'model': optimization.model_name,
        'best_params': optimization.best_params,
        'metric': optimization.metric,
        'n_trials': len(optimization.trials),
        'validation': 'GroupKFold by engine_id',
        'dataset': 'NASA C-MAPSS FD001',
    },
    metrics={f'best_cv_{optimization.metric}': optimization.best_value},
    tags={'project': config['project']['name'], 'phase': '11'},
    artifacts=[trials_path],
    model=optimization.best_model,
    model_artifact_path=tracking_config['model_artifact_path'],
    tracking_uri=f"file:{(project_root / tracking_config['tracking_uri']).resolve()}",
)
print(f'MLflow run ID: {run_id}')